In [1]:
import pandas as pd
import numpy as np

ratings = pd.read_csv(
    "../data/processed/ratings_master.csv"
)

ratings.head()

,User-ID,ISBN,Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [2]:
explicit_ratings = ratings[
    ratings["Rating"] > 0
].copy()

print(explicit_ratings.shape)

(383842, 3)


In [3]:
user_counts = (
    explicit_ratings
    .groupby("User-ID")
    .size()
)

In [4]:
user_counts.describe()

count    68091.000000
mean         5.637191
std         41.742511
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max       6943.000000
dtype: float64

In [5]:
thresholds = [5, 10, 20]

for t in thresholds:

    active_users = user_counts[
        user_counts >= t
    ].index

    subset = explicit_ratings[
        explicit_ratings["User-ID"]
        .isin(active_users)
    ]

    print(f"\nThreshold >= {t}")

    print("Users:",
          subset["User-ID"].nunique())

    print("Books:",
          subset["ISBN"].nunique())

    print("Ratings:",
          len(subset))


Threshold >= 5
Users: 12787
Books: 131372
Ratings: 302218

Threshold >= 10
Users: 6589
Books: 121075
Ratings: 261899

Threshold >= 20
Users: 3305
Books: 108380
Ratings: 217729


In [6]:
datasets = {
    "All": explicit_ratings,
    ">=5": explicit_ratings[
        explicit_ratings["User-ID"].isin(
            user_counts[user_counts >= 5].index
        )
    ],
    ">=10": explicit_ratings[
        explicit_ratings["User-ID"].isin(
            user_counts[user_counts >= 10].index
        )
    ],
    ">=20": explicit_ratings[
        explicit_ratings["User-ID"].isin(
            user_counts[user_counts >= 20].index
        )
    ]
}

for name, df in datasets.items():

    users = df["User-ID"].nunique()
    books = df["ISBN"].nunique()
    ratings = len(df)

    density = (
        ratings / (users * books)
    ) * 100

    print(
        f"{name}: "
        f"{density:.4f}%"
    )

All: 0.0038%
>=5: 0.0180%
>=10: 0.0328%
>=20: 0.0608%


In [7]:
active_users = user_counts[
    user_counts >= 10
].index

ratings_cf = explicit_ratings[
    explicit_ratings["User-ID"].isin(active_users)
].copy()

print(ratings_cf.shape)

(261899, 3)


In [8]:
user2idx = {
    user: idx
    for idx, user in enumerate(
        ratings_cf["User-ID"].unique()
    )
}

idx2user = {
    idx: user
    for user, idx in user2idx.items()
}

In [9]:
book2idx = {
    isbn: idx
    for idx, isbn in enumerate(
        ratings_cf["ISBN"].unique()
    )
}

idx2book = {
    idx: isbn
    for isbn, idx in book2idx.items()
}

In [10]:
ratings_cf["user_idx"] = (
    ratings_cf["User-ID"]
    .map(user2idx)
)

ratings_cf["book_idx"] = (
    ratings_cf["ISBN"]
    .map(book2idx)
)


In [11]:
ratings_cf.head()

,User-ID,ISBN,Rating,user_idx,book_idx
99,276822,0060096195,10,0,0
100,276822,0141310340,9,0,1
101,276822,0142302198,10,0,2
102,276822,0156006065,9,0,3
103,276822,0375821813,9,0,4


In [12]:
ratings_cf.shape

ratings_cf.head()

,User-ID,ISBN,Rating,user_idx,book_idx
99,276822,0060096195,10,0,0
100,276822,0141310340,9,0,1
101,276822,0142302198,10,0,2
102,276822,0156006065,9,0,3
103,276822,0375821813,9,0,4


In [13]:
from sklearn.model_selection import train_test_split

train_rows = []
test_rows = []

for user_id, group in ratings_cf.groupby("user_idx"):

    if len(group) < 2:
        continue

    test_sample = group.sample(
        n=1,
        random_state=42
    )

    train_sample = group.drop(
        test_sample.index
    )

    train_rows.append(train_sample)
    test_rows.append(test_sample)

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows)

In [14]:
print("Train:", train_df.shape)
print("Test:", test_df.shape)

print(
    "Users in Train:",
    train_df["user_idx"].nunique()
)

print(
    "Users in Test:",
    test_df["user_idx"].nunique()
)

Train: (255310, 5)
Test: (6589, 5)
Users in Train: 6589
Users in Test: 6589


In [15]:
num_users = ratings_cf["user_idx"].nunique()
num_books = ratings_cf["book_idx"].nunique()

In [16]:
train_df.shape
test_df.shape

train_df["user_idx"].nunique()
test_df["user_idx"].nunique()

6589

In [17]:
print("Users:", ratings_cf["user_idx"].nunique())
print("Books:", ratings_cf["book_idx"].nunique())

Users: 6589
Books: 121075


In [18]:
import torch

train_users = torch.tensor(
    train_df["user_idx"].values,
    dtype=torch.long
)

train_books = torch.tensor(
    train_df["book_idx"].values,
    dtype=torch.long
)

train_ratings = torch.tensor(
    train_df["Rating"].values,
    dtype=torch.float32
)

In [19]:
test_users = torch.tensor(
    test_df["user_idx"].values,
    dtype=torch.long
)

test_books = torch.tensor(
    test_df["book_idx"].values,
    dtype=torch.long
)

test_ratings = torch.tensor(
    test_df["Rating"].values,
    dtype=torch.float32
)

In [ ]:
import torch
import torch.nn as nn


class MatrixFactorization(nn.Module):

    def __init__(
        self,
        num_users,
        num_books,
        embedding_dim=50
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.book_embedding = nn.Embedding(
            num_books,
            embedding_dim
        )

        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.book_bias = nn.Embedding(
            num_books,
            1
        )

    def forward(
        self,
        users,
        books
    ):

        user_vecs = self.user_embedding(users)

        book_vecs = self.book_embedding(books)

        interaction = (
            user_vecs * book_vecs
        ).sum(dim=1)

        user_bias = (
            self.user_bias(users)
            .squeeze()
        )

        book_bias = (
            self.book_bias(books)
            .squeeze()
        )

        return (
            interaction
            + user_bias
            + book_bias
        )

In [24]:
model = MatrixFactorization(
    num_users=6589,
    num_books=121075,
    embedding_dim=50
)

In [25]:
criterion = nn.MSELoss()

In [26]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [27]:
epochs = 10

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    predictions = model(
        train_users,
        train_books
    )

    loss = criterion(
        predictions,
        train_ratings
    )

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1}/{epochs}",
        f"Loss: {loss.item():.4f}"
    )

Epoch 1/10 Loss: 115.9987
Epoch 2/10 Loss: 115.3574
Epoch 3/10 Loss: 114.7193
Epoch 4/10 Loss: 114.0844
Epoch 5/10 Loss: 113.4527
Epoch 6/10 Loss: 112.8242
Epoch 7/10 Loss: 112.1991
Epoch 8/10 Loss: 111.5772
Epoch 9/10 Loss: 110.9587
Epoch 10/10 Loss: 110.3436


In [28]:
model.eval()

with torch.no_grad():

    preds = model(
        test_users,
        test_books
    )

    rmse = torch.sqrt(
        ((preds - test_ratings) ** 2).mean()
    )

print("RMSE:", rmse.item())

RMSE: 10.382558822631836


In [29]:
train_ratings.mean()

tensor(7.7198)

In [30]:
with torch.no_grad():

    preds = model(
        test_users,
        test_books
    )

print(preds[:20])

tensor([ -9.6230,   7.5774,   4.4068,  15.6845,   8.9774, -20.7985,   6.9213,
          5.6167,  -4.9992,  -7.5084,   2.8666,   8.4905,  -3.9690,   0.2481,
          0.5734,   5.0309,   5.2997,   1.5374,  -0.0274,   1.8446])


In [31]:
print(test_ratings[:20])

tensor([ 8.,  8.,  5.,  5.,  6., 10.,  6., 10.,  6.,  6., 10.,  8.,  5.,  9.,
         7., 10.,  7.,  7.,  8., 10.])


In [32]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

train_dataset = TensorDataset(
    train_users,
    train_books,
    train_ratings
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True
)

In [33]:
global_mean = train_ratings.mean().item()

In [34]:
class MatrixFactorization(nn.Module):

    def __init__(
        self,
        num_users,
        num_books,
        global_mean,
        embedding_dim=50
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.book_embedding = nn.Embedding(
            num_books,
            embedding_dim
        )

        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.book_bias = nn.Embedding(
            num_books,
            1
        )

        self.global_mean = global_mean

    def forward(self, users, books):

        user_vecs = self.user_embedding(users)

        book_vecs = self.book_embedding(books)

        interaction = (
            user_vecs * book_vecs
        ).sum(dim=1)

        user_bias = (
            self.user_bias(users)
            .squeeze()
        )

        book_bias = (
            self.book_bias(books)
            .squeeze()
        )

        pred = (
            self.global_mean
            + interaction
            + user_bias
            + book_bias
        )

        return pred

In [35]:
model = MatrixFactorization(
    num_users=6589,
    num_books=121075,
    global_mean=global_mean,
    embedding_dim=50
)

In [36]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [37]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        optimizer.zero_grad()

        preds = model(
            users,
            books
        )

        loss = criterion(
            preds,
            ratings
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}",
        f"Loss: {total_loss:.4f}"
    )

Epoch 1 Loss: 6528.7860
Epoch 2 Loss: 5857.1964
Epoch 3 Loss: 5230.3598
Epoch 4 Loss: 4671.4709
Epoch 5 Loss: 4178.1910
Epoch 6 Loss: 3740.0793
Epoch 7 Loss: 3352.6541
Epoch 8 Loss: 3007.5880
Epoch 9 Loss: 2701.0387
Epoch 10 Loss: 2427.4400


In [38]:
model.eval()

with torch.no_grad():

    preds = model(
        test_users,
        test_books
    )

    rmse = torch.sqrt(
        ((preds - test_ratings) ** 2).mean()
    )

print("RMSE:", rmse.item())

RMSE: 5.6470794677734375


In [39]:
print(preds[:20])
print(test_ratings[:20])

tensor([ 8.6278,  2.7472,  8.7905, 11.6707, 16.7259,  9.3194,  4.7875, -0.7867,
         5.7578, 13.7369, 12.6855,  5.6326,  7.3815, 10.2349, 16.5962,  7.3888,
        -4.2474,  1.8989,  5.9518, 18.7672])
tensor([ 8.,  8.,  5.,  5.,  6., 10.,  6., 10.,  6.,  6., 10.,  8.,  5.,  9.,
         7., 10.,  7.,  7.,  8., 10.])


In [40]:
preds_clipped = torch.clamp(
    preds,
    min=1,
    max=10
)

rmse_clipped = torch.sqrt(
    ((preds_clipped - test_ratings) ** 2).mean()
)

print("Clipped RMSE:",
      rmse_clipped.item())

Clipped RMSE: 3.684438705444336


In [41]:
book_counts = (
    ratings_cf
    .groupby("ISBN")
    .size()
)

book_counts.describe()

count    121075.000000
mean          2.163114
std           4.843571
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         330.000000
dtype: float64

In [42]:
active_books = book_counts[
    book_counts >= 5
].index

In [43]:
ratings_dense = ratings_cf[
    ratings_cf["ISBN"]
    .isin(active_books)
].copy()

In [44]:
print("Users:",
      ratings_dense["User-ID"].nunique())

print("Books:",
      ratings_dense["ISBN"].nunique())

print("Ratings:",
      len(ratings_dense))

Users: 6436
Books: 9009
Ratings: 108344


In [45]:
book_counts.describe()


count    121075.000000
mean          2.163114
std           4.843571
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         330.000000
dtype: float64

In [46]:
active_books = book_counts[
    book_counts >= 5
].index

ratings_dense = ratings_cf[
    ratings_cf["ISBN"].isin(active_books)
]

print(ratings_dense["User-ID"].nunique())
print(ratings_dense["ISBN"].nunique())
print(len(ratings_dense))

6436
9009
108344


In [47]:
dense_user_counts = (
    ratings_dense
    .groupby("User-ID")
    .size()
)

dense_book_counts = (
    ratings_dense
    .groupby("ISBN")
    .size()
)

print("Users")
print(dense_user_counts.describe())

print("\nBooks")
print(dense_book_counts.describe())

Users
count    6436.000000
mean       16.834058
std        39.303941
min         1.000000
25%         5.000000
50%         9.000000
75%        18.000000
max      2436.000000
dtype: float64

Books
count    9009.000000
mean       12.026196
std        14.263046
min         5.000000
25%         6.000000
50%         8.000000
75%        13.000000
max       330.000000
dtype: float64


In [48]:
user2idx = {
    user: idx
    for idx, user in enumerate(
        ratings_dense["User-ID"].unique()
    )
}

In [49]:
book2idx = {
    isbn: idx
    for idx, isbn in enumerate(
        ratings_dense["ISBN"].unique()
    )
}

In [50]:
ratings_dense["user_idx"] = (
    ratings_dense["User-ID"]
    .map(user2idx)
)

ratings_dense["book_idx"] = (
    ratings_dense["ISBN"]
    .map(book2idx)
)

In [51]:
train_rows = []
test_rows = []

for user_id, group in ratings_dense.groupby("user_idx"):

    if len(group) < 2:
        continue

    test_sample = group.sample(
        n=1,
        random_state=42
    )

    train_sample = group.drop(
        test_sample.index
    )

    train_rows.append(train_sample)
    test_rows.append(test_sample)

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows)

In [52]:
print(train_df.shape)
print(test_df.shape)

print(
    train_df["user_idx"].nunique()
)

print(
    test_df["user_idx"].nunique()
)

(101908, 5)
(6197, 5)
6197
6197


In [53]:
num_users = ratings_dense["user_idx"].nunique()
num_books = ratings_dense["book_idx"].nunique()

density = (
    len(ratings_dense)
    /
    (num_users * num_books)
) * 100

print("Users:", num_users)
print("Books:", num_books)
print("Density:", density)

Users: 6436
Books: 9009
Density: 0.18685823533555043


In [54]:
train_users = torch.tensor(
    train_df["user_idx"].values,
    dtype=torch.long
)

train_books = torch.tensor(
    train_df["book_idx"].values,
    dtype=torch.long
)

train_ratings = torch.tensor(
    train_df["Rating"].values,
    dtype=torch.float32
)

In [55]:
test_users = torch.tensor(
    test_df["user_idx"].values,
    dtype=torch.long
)

test_books = torch.tensor(
    test_df["book_idx"].values,
    dtype=torch.long
)

test_ratings = torch.tensor(
    test_df["Rating"].values,
    dtype=torch.float32
)

In [56]:
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

train_dataset = TensorDataset(
    train_users,
    train_books,
    train_ratings
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True
)

In [57]:
global_mean = (
    train_ratings.mean()
    .item()
)

print(global_mean)

7.838815212249756


In [58]:
model = MatrixFactorization(
    num_users=ratings_dense["user_idx"].nunique(),
    num_books=ratings_dense["book_idx"].nunique(),
    global_mean=global_mean,
    embedding_dim=50
)

In [59]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [60]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        optimizer.zero_grad()

        preds = model(users, books)

        loss = criterion(preds, ratings)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Avg Loss: {avg_loss:.4f}"
    )

Epoch 1/10 | Avg Loss: 53.4917
Epoch 2/10 | Avg Loss: 50.3421
Epoch 3/10 | Avg Loss: 47.5253
Epoch 4/10 | Avg Loss: 44.8410
Epoch 5/10 | Avg Loss: 42.3098
Epoch 6/10 | Avg Loss: 39.9226
Epoch 7/10 | Avg Loss: 37.6818
Epoch 8/10 | Avg Loss: 35.5661
Epoch 9/10 | Avg Loss: 33.6006
Epoch 10/10 | Avg Loss: 31.7326


In [61]:
model.eval()

with torch.no_grad():

    preds = model(
        test_users,
        test_books
    )

    rmse = torch.sqrt(
        ((preds - test_ratings) ** 2).mean()
    )

print("RMSE:", rmse.item())

RMSE: 6.91235876083374


In [62]:
preds_clipped = torch.clamp(
    preds,
    min=1,
    max=10
)

rmse_clipped = torch.sqrt(
    ((preds_clipped - test_ratings) ** 2).mean()
)

print("Clipped RMSE:",
      rmse_clipped.item())

Clipped RMSE: 4.115969181060791


In [63]:
print(preds[:20])
print(test_ratings[:20])

tensor([ 1.1323e+01,  6.4812e+00,  1.8092e+01,  1.3874e+01, -4.0064e-03,
         1.6482e+01, -2.2813e+00,  1.2604e+01,  5.1554e+00,  8.9104e+00,
         9.1544e+00,  7.2658e+00,  1.1718e+01,  1.4896e+01,  1.0826e+01,
         4.8693e+00,  1.8076e+00,  1.5294e+01,  6.3268e+00,  3.1977e+00])
tensor([10.,  8.,  6.,  2.,  9.,  8.,  6.,  5., 10.,  8.,  4.,  8.,  7., 10.,
         7.,  7.,  4.,  5., 10.,  7.])


In [64]:
with torch.no_grad():

    user_norms = (
        model.user_embedding.weight.norm(dim=1)
    )

    book_norms = (
        model.book_embedding.weight.norm(dim=1)
    )

print(user_norms.mean())
print(book_norms.mean())

tensor(6.7125)
tensor(6.8338)


In [65]:
global_mean = train_ratings.mean()

baseline_rmse = torch.sqrt(
    (
        (test_ratings - global_mean) ** 2
    ).mean()
)

print(baseline_rmse)

tensor(1.8307)


In [67]:
import torch
import torch.nn as nn

class BiasModel(nn.Module):

    def __init__(self, num_users, num_books, global_mean):
        super().__init__()

        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.book_bias = nn.Embedding(
            num_books,
            1
        )

        self.global_mean = global_mean

    def forward(self, users, books):

        user_bias = (
            self.user_bias(users)
            .squeeze()
        )

        book_bias = (
            self.book_bias(books)
            .squeeze()
        )

        return (
            self.global_mean
            + user_bias
            + book_bias
        )

In [68]:
global_mean = train_ratings.mean().item()

bias_model = BiasModel(
    num_users=ratings_dense["user_idx"].nunique(),
    num_books=ratings_dense["book_idx"].nunique(),
    global_mean=global_mean
)

In [69]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    bias_model.parameters(),
    lr=0.001
)

In [70]:
epochs = 10

for epoch in range(epochs):

    bias_model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        optimizer.zero_grad()

        preds = bias_model(
            users,
            books
        )

        loss = criterion(
            preds,
            ratings
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Avg Loss: {avg_loss:.4f}"
    )

Epoch 1/10 | Avg Loss: 5.2535
Epoch 2/10 | Avg Loss: 5.1705
Epoch 3/10 | Avg Loss: 5.0899
Epoch 4/10 | Avg Loss: 5.0107
Epoch 5/10 | Avg Loss: 4.9329
Epoch 6/10 | Avg Loss: 4.8578
Epoch 7/10 | Avg Loss: 4.7837
Epoch 8/10 | Avg Loss: 4.7109
Epoch 9/10 | Avg Loss: 4.6434
Epoch 10/10 | Avg Loss: 4.5742


In [71]:
bias_model.eval()

with torch.no_grad():

    preds = bias_model(
        test_users,
        test_books
    )

    rmse = torch.sqrt(
        ((preds - test_ratings) ** 2).mean()
    )

print("RMSE:", rmse.item())

RMSE: 2.1972897052764893


In [72]:
popular_books = (
    train_df
    .groupby("book_idx")
    .size()
    .sort_values(ascending=False)
)

popular_books.head(10)

book_idx
438    311
11     256
71     178
464    172
751    167
490    164
215    150
366    148
283    147
376    140
dtype: int64

In [73]:
idx2book = {
    idx: isbn
    for isbn, idx in book2idx.items()
}

In [75]:
top_books = []

for idx in popular_books.head(10).index:

    isbn = idx2book[idx]

    title = books_master.loc[
        books_master["ISBN"] == isbn,
        "Book-Title"
    ].iloc[0]

    top_books.append(title)

top_books

NameError: name 'books_master' is not defined

In [78]:
whos

Variable              Type                   Data/Info
------------------------------------------------------
BiasModel             type                   <class '__main__.BiasModel'>
DataLoader            type                   <class 'torch.utils.data.dataloader.DataLoader'>
MatrixFactorization   type                   <class '__main__.MatrixFactorization'>
TensorDataset         type                   <class 'torch.utils.data.dataset.TensorDataset'>
active_books          Index                  Index(['0002251760', '000<...>name='ISBN', length=9009)
active_users          Index                  Index([   242,    243,   <...>e='User-ID', length=6589)
avg_loss              float                  4.574207601547241
baseline_rmse         Tensor                 tensor(1.8307)
bias_model            BiasModel              BiasModel(\n  (user_bias)<...>s): Embedding(9009, 1)\n)
book2idx              dict                   n=9009
book_counts           Series                 Shape: (121075,)
book

In [79]:
import pandas as pd

books_df = pd.read_csv(
    "../data/processed/books_master.csv"
)

print(books_df.shape)
books_df.head()

(271360, 13)


C:\Users\anshu\AppData\Local\Temp\ipykernel_15464\4007194975.py:3: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df = pd.read_csv(


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,description,categories,language,page_count,thumbnail
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,NaN,NaN,NaN,NaN,NaN
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,NaN,NaN,NaN,NaN,NaN
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,NaN,NaN,NaN,NaN,NaN
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,NaN,NaN,NaN,NaN,NaN
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,NaN,NaN,NaN,NaN,NaN


In [80]:
books_df.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'description',
       'categories', 'language', 'page_count', 'thumbnail'],
      dtype='str')

In [82]:
idx2book = {
    idx: isbn
    for isbn, idx in book2idx.items()
}

top_books = []

for idx in popular_books.head(10).index:

    isbn = idx2book[idx]

    title = books_df.loc[
        books_df["ISBN"] == isbn,
        "Book-Title"
    ].iloc[0]

    author = books_df.loc[
        books_df["ISBN"] == isbn,
        "Book-Author"
    ].iloc[0]

    top_books.append(
        (title, author)
    )

top_books

[('The Lovely Bones: A Novel', 'Alice Sebold'),
 ('The Da Vinci Code', 'Dan Brown'),
 ('The Red Tent (Bestselling Backlist)', 'Anita Diamant'),
 ("Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))",
  'J. K. Rowling'),
 ('Wild Animus', 'Rich Shapero'),
 ('The Secret Life of Bees', 'Sue Monk Kidd'),
 ("Where the Heart Is (Oprah's Book Club (Paperback))", 'Billie Letts'),
 ('Harry Potter and the Order of the Phoenix (Book 5)', 'J. K. Rowling'),
 ('Interview with the Vampire', 'Anne Rice'),
 ('Angels &amp; Demons', 'Dan Brown')]

In [83]:
top10_popular = (
    popular_books
    .head(10)
    .index
    .tolist()
)

print(top10_popular)

[438, 11, 71, 464, 751, 490, 215, 366, 283, 376]


In [84]:
hits = 0

for _, row in test_df.iterrows():

    true_book = row["book_idx"]

    if true_book in top10_popular:

        hits += 1

hit10 = hits / len(test_df)

print("Hit@10:", hit10)

Hit@10: 0.020816524124576408


In [85]:
user_book_matrix = (
    train_df
    .pivot_table(
        index="user_idx",
        columns="book_idx",
        values="Rating",
        fill_value=0
    )
)

print(user_book_matrix.shape)

(6197, 9009)


In [86]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(
    user_book_matrix.T
)

print(item_similarity.shape)

(9009, 9009)


In [87]:
item_similarity.nbytes / (1024**2)

619.2175369262695

In [88]:
import pandas as pd

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_book_matrix.columns,
    columns=user_book_matrix.columns
)

item_similarity_df.shape

(9009, 9009)

In [89]:
sample_book = user_book_matrix.columns[0]

sample_book

np.int64(0)

In [90]:
item_similarity_df.loc[sample_book] \
                  .sort_values(ascending=False) \
                  .head(10)

book_idx
0       1.000000
5783    0.256731
2126    0.243364
1999    0.229005
2906    0.211504
1787    0.207928
2284    0.203124
2292    0.200447
2280    0.197737
2281    0.195148
Name: 0, dtype: float64

In [91]:
idx_to_book = (
    books_filtered
    .reset_index(drop=True)
)

NameError: name 'books_filtered' is not defined

In [92]:
train_df.head()

,User-ID,ISBN,Rating,user_idx,book_idx
103,276822,0375821813,9,0,1
109,276822,0786817070,10,0,2
152,276847,3442446414,10,1,4
168,276847,3551551677,10,1,5
169,276847,3551551685,10,1,6


In [93]:
ratings_master.head()

NameError: name 'ratings_master' is not defined

In [94]:
%whos DataFrame


Variable             Type         Data/Info
-------------------------------------------
books_df             DataFrame    Shape: (271360, 13)
df                   DataFrame    Shape: (217729, 3)
explicit_ratings     DataFrame    Shape: (383842, 3)
group                DataFrame    Shape: (13, 5)
item_similarity_df   DataFrame    Shape: (9009, 9009)
ratings_cf           DataFrame    Shape: (261899, 5)
ratings_dense        DataFrame    Shape: (108344, 5)
subset               DataFrame    Shape: (217729, 3)
test_df              DataFrame    Shape: (6197, 5)
test_sample          DataFrame    Shape: (1, 5)
train_df             DataFrame    Shape: (101908, 5)
train_sample         DataFrame    Shape: (12, 5)
user_book_matrix     DataFrame    Shape: (6197, 9009)


In [96]:
books_df.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'description',
       'categories', 'language', 'page_count', 'thumbnail'],
      dtype='str')

In [97]:
books_df.head(2)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,description,categories,language,page_count,thumbnail
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,NaN,NaN,NaN,NaN,NaN
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,NaN,NaN,NaN,NaN,NaN


In [98]:
sample_user = 0

recs = recommend_books(
    sample_user,
    train_df,
    item_similarity_df,
    top_n=10
)

recs

NameError: name 'recommend_books' is not defined

In [99]:
import pandas as pd

def recommend_books(
    user_id,
    train_df,
    item_similarity_df,
    top_n=10
):

    user_ratings = train_df[
        train_df["user_idx"] == user_id
    ]

    scores = {}

    for _, row in user_ratings.iterrows():

        book = row["book_idx"]
        rating = row["Rating"]

        similar_books = item_similarity_df[book]

        for sim_book, sim_score in similar_books.items():

            if sim_book == book:
                continue

            scores[sim_book] = (
                scores.get(sim_book, 0)
                + sim_score * rating
            )

    already_read = set(
        user_ratings["book_idx"]
    )

    recommendations = (
        pd.Series(scores)
        .drop(labels=already_read, errors="ignore")
        .sort_values(ascending=False)
        .head(top_n)
    )

    return recommendations

In [100]:
sample_user = 0

recs = recommend_books(
    sample_user,
    train_df,
    item_similarity_df,
    top_n=10
)

recs

4458    2.415546
8463    2.279259
8616    2.222449
3899    2.040078
5253    1.939707
6412    1.935050
8625    1.908318
5349    1.876929
1087    1.871589
5175    1.839932
dtype: float64

In [101]:
books_df.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'description',
       'categories', 'language', 'page_count', 'thumbnail'],
      dtype='str')

In [102]:
recs

4458    2.415546
8463    2.279259
8616    2.222449
3899    2.040078
5253    1.939707
6412    1.935050
8625    1.908318
5349    1.876929
1087    1.871589
5175    1.839932
dtype: float64

In [103]:
ratings_dense.head()

,User-ID,ISBN,Rating,user_idx,book_idx
99,276822,0060096195,10,0,0
103,276822,0375821813,9,0,1
109,276822,0786817070,10,0,2
137,276847,3404148576,8,1,3
152,276847,3442446414,10,1,4


In [104]:
ratings_dense.columns

Index(['User-ID', 'ISBN', 'Rating', 'user_idx', 'book_idx'], dtype='str')

In [105]:
train_df[
    train_df["user_idx"] == 0
]


,User-ID,ISBN,Rating,user_idx,book_idx
103,276822,0375821813,9,0,1
109,276822,0786817070,10,0,2


In [106]:
book_lookup = (
    ratings_dense[["book_idx", "ISBN"]]
    .drop_duplicates()
)

book_lookup.head()

,book_idx,ISBN
99,0,0060096195
103,1,0375821813
109,2,0786817070
137,3,3404148576
152,4,3442446414


In [107]:
recs

4458    2.415546
8463    2.279259
8616    2.222449
3899    2.040078
5253    1.939707
6412    1.935050
8625    1.908318
5349    1.876929
1087    1.871589
5175    1.839932
dtype: float64

In [108]:
rec_df = pd.DataFrame({
    "book_idx": recs.index,
    "score": recs.values
})

rec_df = rec_df.merge(
    book_lookup,
    on="book_idx",
    how="left"
)

rec_df = rec_df.merge(
    books_df[[
        "ISBN",
        "Book-Title",
        "Book-Author"
    ]],
    on="ISBN",
    how="left"
)

rec_df

,book_idx,score,ISBN,Book-Title,Book-Author
0,4458,2.415546,0786808551,"The Arctic Incident (Artemis Fowl, Book 2)",Eoin Colfer
1,8463,2.279259,0805073337,What Was She Thinking?: Notes on a Scandal: A ...,Zoe Heller
2,8616,2.222449,0786851473,Artemis Fowl: The Arctic Incident - Book #2 (A...,Eoin Colfer
3,3899,2.040078,0590407600,Magic School Bus: Inside the Earth (Magic Scho...,Joanna Cole
4,5253,1.939707,0064408639,The Austere Academy (A Series of Unfortunate E...,Lemony Snicket
5,6412,1.935050,0590484125,The Magic School Bus in the Haunted Museum: A ...,Joanna Cole
6,8625,1.908318,0345382293,If I'd Killed Him When I Met Him...: An Elizab...,Sharyn McCrumb
7,5349,1.876929,0140185003,The Quiet American (Penguin Twentieth Century ...,Graham Greene
8,1087,1.871589,067975833X,Confederates in the Attic : Dispatches from th...,TONY HORWITZ
9,5175,1.839932,0375709150,Driving over Lemons: An Optimist in Spain (Vin...,Chris Stewart


In [109]:
user0 = train_df[
    train_df["user_idx"] == 0
]

user0 = user0.merge(
    books_df[[
        "ISBN",
        "Book-Title",
        "Book-Author"
    ]],
    on="ISBN",
    how="left"
)

user0[[
    "Book-Title",
    "Book-Author",
    "Rating"
]]

,Book-Title,Book-Author,Rating
0,Hoot (Newbery Honor Book),CARL HIAASEN,9
1,"Artemis Fowl (Artemis Fowl, Book 1)",Eoin Colfer,10


In [111]:
len(test_df)

6197

In [112]:
def recommend_top_n(
    user_id,
    train_df,
    item_similarity_df,
    top_n=10
):

    user_ratings = train_df[
        train_df["user_idx"] == user_id
    ]

    scores = {}

    for _, row in user_ratings.iterrows():

        book = row["book_idx"]
        rating = row["Rating"]

        similar_books = item_similarity_df[book]

        for sim_book, sim_score in similar_books.items():

            if sim_book == book:
                continue

            scores[sim_book] = (
                scores.get(sim_book, 0)
                + sim_score * rating
            )

    already_read = set(
        user_ratings["book_idx"]
    )

    recommendations = (
        pd.Series(scores)
        .drop(labels=already_read, errors="ignore")
        .sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )

    return recommendations

In [113]:
recommend_top_n(
    0,
    train_df,
    item_similarity_df,
    top_n=10
)


[4458, 8463, 8616, 3899, 5253, 6412, 8625, 5349, 1087, 5175]

In [114]:
hits = 0

for _, row in test_df.iterrows():

    user_id = row["user_idx"]
    true_book = row["book_idx"]

    recommendations = recommend_top_n(
        user_id,
        train_df,
        item_similarity_df,
        top_n=10
    )

    if true_book in recommendations:
        hits += 1

hit10 = hits / len(test_df)

print("Hits:", hits)
print("Hit@10:", hit10)

Hits: 324
Hit@10: 0.052283362917540745


In [115]:
coverage = (
    test_df["book_idx"]
    .nunique()
)

print("Unique test books:", coverage)

Unique test books: 3752


In [116]:
len(set(test_df["book_idx"]))

3752

In [117]:
def recommend_top_n_k(
    user_id,
    train_df,
    item_similarity_df,
    top_n=10,
    k=50
):

    user_ratings = train_df[
        train_df["user_idx"] == user_id
    ]

    scores = {}

    for _, row in user_ratings.iterrows():

        book = row["book_idx"]
        rating = row["Rating"]

        neighbors = (
            item_similarity_df[book]
            .sort_values(ascending=False)
            .iloc[1:k+1]
        )

        for sim_book, sim_score in neighbors.items():

            scores[sim_book] = (
                scores.get(sim_book, 0)
                + sim_score * rating
            )

    already_read = set(
        user_ratings["book_idx"]
    )

    recommendations = (
        pd.Series(scores)
        .drop(labels=already_read, errors="ignore")
        .sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )

    return recommendations

In [118]:
import pandas as pd

def recommend_top_n_k(
    user_id,
    train_df,
    item_similarity_df,
    top_n=10,
    k=50
):

    user_ratings = train_df[
        train_df["user_idx"] == user_id
    ]

    scores = {}

    for _, row in user_ratings.iterrows():

        book = row["book_idx"]
        rating = row["Rating"]

        # Get only top-k similar books
        neighbors = (
            item_similarity_df[book]
            .sort_values(ascending=False)
            .iloc[1:k+1]      # skip self-similarity
        )

        for sim_book, sim_score in neighbors.items():

            scores[sim_book] = (
                scores.get(sim_book, 0)
                + sim_score * rating
            )

    already_read = set(
        user_ratings["book_idx"]
    )

    recommendations = (
        pd.Series(scores)
        .drop(labels=already_read, errors="ignore")
        .sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )

    return recommendations

In [119]:
hits = 0

for _, row in test_df.iterrows():

    user_id = row["user_idx"]
    true_book = row["book_idx"]

    recommendations = recommend_top_n_k(
        user_id=user_id,
        train_df=train_df,
        item_similarity_df=item_similarity_df,
        top_n=10,
        k=50
    )

    if true_book in recommendations:
        hits += 1

hit10 = hits / len(test_df)

print("Hits:", hits)
print("Hit@10:", hit10)

Hits: 330
Hit@10: 0.05325157334193965


In [120]:
k_values = [10, 25, 50, 100, 200]

results = []

for k in k_values:

    hits = 0

    for _, row in test_df.iterrows():

        user_id = row["user_idx"]
        true_book = row["book_idx"]

        recommendations = recommend_top_n_k(
            user_id=user_id,
            train_df=train_df,
            item_similarity_df=item_similarity_df,
            top_n=10,
            k=k
        )

        if true_book in recommendations:
            hits += 1

    hit10 = hits / len(test_df)

    results.append({
        "K": k,
        "Hits": hits,
        "Hit@10": hit10
    })

    print(f"K={k} | Hits={hits} | Hit@10={hit10:.4f}")

K=10 | Hits=368 | Hit@10=0.0594
K=25 | Hits=346 | Hit@10=0.0558
K=50 | Hits=330 | Hit@10=0.0533
K=100 | Hits=312 | Hit@10=0.0503
K=200 | Hits=301 | Hit@10=0.0486


In [121]:
results_df = pd.DataFrame(results)

results_df

,K,Hits,Hit@10
0,10,368,0.059384
1,25,346,0.055833
2,50,330,0.053252
3,100,312,0.050347
4,200,301,0.048572
